# EXP-2026-005 / Q5-B-0 — S 하위분류 키 복구 (quest51)

**상태: DESIGN / RESULT NOT RUN.** 아래 셀을 실행하기 전까지 이 문서의 어떤 숫자도 결과가 아니다.

- **ANALYSIS ONLY / NO TRAINING** — 학습하지 않는다. 저장된 예측·처리 완료 배열만 읽는다. GPU가 필요 없다.
- 이 실험이 하는 일은 하나다: Q5-A가 측정하지 못한 다섯 번째 블록 `B_SUBTYPE`을 **되살릴 수 있는지 재보는 것**. Q5-A는 `t`가 annotation sample index가 아니라 `.atr` 조인이 1.9%(우연 수준)에 그쳐 이 블록을 비운 채 `UNRESOLVED`(D5)로 끝났다.
- 여기서 말할 수 있는 것은 `failure-associated factor`(**실패 연관 요인**)까지다. `원인`은 요인 하나만 바꾸는 개입과 음성대조군으로만 검증한다.
- **residual CNN 경로는 closed**(Q4-O NO-GO, Q4-Q mechanism+utility fail)이며 재개하거나 변형을 제안하지 않는다. **INCART rescue run도 하지 않는다.**
- gate가 `NO_GO_SUBTYPE_CLOSED`면 `B_SUBTYPE`을 **영구 미측정**으로 종결한다. 추정으로 채우지 않고, 이것을 만들려고 재학습하지 않는다. **그것도 결과다.**
- **Q5-B-1(개입 pilot)은 이 notebook에서 구현하지 않는다.**

spec: `experiments/specs/EXP-2026-005-q5b0-subtype-key-recovery.md`

## mode 실행 순서 (정확히 하나만 활성)

| 순서 | mode | 하는 일 |
|---|---|---|
| ① | — | 셀 2: repo 준비 + commit SHA + Q5-A / Q5-B-0 회귀 테스트 |
| ② | — | 셀 3: Drive mount + 입력 경로 |
| ③ | `RECOVER` | 셀 4: S beat 조인 + 음성대조군 + gate (재분석 없음) |
| ④ | `RECOVER` | 셀 5: gate 결과 확인 — `NO_GO`면 **여기서 끝**이고 그게 결론이다 |
| ⑤ | `REANALYZE` | 셀 6: GO일 때만, Q5-A의 `run_atlas`를 그대로 다시 실행(5개 블록) |
| ⑥ | `REPORT` | 셀 7: 저장 bundle만 다시 표시 (재계산 없음) |

기본값은 `DESIGN`(데이터 접근 없음).

## 입력 (Drive)

- **동결 atlas cohort**: `MyDrive/mitbih/mamba_data.npz` (file id `1p3HvC_bnbiQlEanFOVIvVdejy60W0tho`) — Q5-A가 쓴 그 파일. `pid`·`y`·`t`(검증된 초)에서 RR을 만든다
- **symbol source**: `MyDrive/mitbih/ecg_multi.npz` (file id `1aSj_1jvS_W2iruVnORIG6DTVuHobzNzq`) — `sym`(원 annotation symbol)을 가진 유일한 파일. waveform(`beat`)은 **읽지 않는다**
- **baseline**: Q5-A와 동일하게 `MyDrive/mitbih/baseline_pkgs/` 의 V10/V10_BASE/V9/V9_BASE (재분석 단계에서만 필요)

조인 키는 `(pre_rr, post_rr)` 초 단위 — **beat 자신의 성질만** 쓴다(이웃 RR을 넣으면 행 순서에 의존하게 되어 "pool을 섞어도 같은 결과"를 증명할 수 없다). symbol은 매칭에 쓰지 않으므로 "매칭된 beat의 symbol이 A/a/J/S에 드는가"가 **독립 검증**이 된다.


In [ ]:
# ── 실행 설정 (정확히 하나의 mode) ────────────────────────────────────────────
VALID_MODES = ("DESIGN", "RECOVER", "REANALYZE", "REPORT")
MODE = "DESIGN"          # DESIGN -> RECOVER -> (GO면) REANALYZE -> REPORT
assert MODE in VALID_MODES, f"MODE must be one of {VALID_MODES}"

BRANCH = "main"          # 이 notebook이 쓸 repo 브랜치
NEED_Q5B0 = 5            # 이 notebook이 요구하는 최소 q5b0 모듈 버전
NEED_Q5A = 8             # 재분석은 인수된 Q5-A(v8) 위에서만 돈다

# Q5-B-0(strict) / Q5-B-0b(tie) — 어느 사전등록으로 돌릴지 고른다.
#   "strict_identity"      = Q5-B-0 원안 (모호하면 미매칭). 기록된 NO-GO 재현용.
#   "tie_symbol_agreement" = Q5-B-0b (동점 집합이 한 symbol이면 부여).
TIE_MODE = "strict_identity"

REPORT_RUN = ""          # REPORT mode에서 읽을 run 폴더 (비우면 최신)
print("mode:", MODE)


In [ ]:
# ── 셀 2: repo 준비 + 회귀 테스트 ─────────────────────────────────────────────
import os, subprocess, sys

REPO = "/content/my-github-test"
os.chdir("/content")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone",
                    "https://github.com/ehdbddl06001-ui/my-github-test.git"],
                   check=True)
os.chdir(REPO)
subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)
print("commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True).stdout.strip())

# 오래된 import가 남아 있으면 stale 모듈로 잘못된 결과가 나온다 -> 강제로 비운다
for _m in [m for m in list(sys.modules)
           if m.startswith(("q5a_", "q5b0_", "q4o_", "q4p_", "q4q_"))]:
    sys.modules.pop(_m, None)
sys.path.insert(0, os.path.join(REPO, "mit-bih"))

import q5a_patient_failure_atlas as QA
import q5b0_subtype_key_recovery as QB

assert QA.MODULE_VERSION >= NEED_Q5A, (
    f"stale module: q5a v{QA.MODULE_VERSION} < v{NEED_Q5A}. BRANCH가 맞는지 "
    "확인하고 Restart runtime 후 다시 실행")
assert QB.MODULE_VERSION >= NEED_Q5B0, (
    f"stale module: q5b0 v{QB.MODULE_VERSION} < v{NEED_Q5B0}. BRANCH가 맞는지 "
    "확인하고 Restart runtime 후 다시 실행")
print("q5a v%d  <- %s" % (QA.MODULE_VERSION, QA.__file__))
print("q5b0 v%d <- %s" % (QB.MODULE_VERSION, QB.__file__))

for suite in ("test_q4o_leakage_free_residual", "test_q4p_best_epoch_zero_diagnostic",
              "test_q4q_transportability_replication",
              "test_q5a_patient_failure_atlas", "test_q5b0_subtype_key_recovery"):
    r = subprocess.run([sys.executable, f"mit-bih/{suite}.py"],
                       capture_output=True, text=True)
    tail = [l for l in r.stdout.splitlines() if l.startswith("passed ")]
    print(f"{suite}: {tail[-1] if tail else 'NO RESULT'}"
          + ("" if r.returncode == 0 else "   <-- FAILED"))
    assert r.returncode == 0, f"{suite} failed — 여기서 멈춘다"


In [ ]:
# ── 셀 3: Drive mount + 입력 경로 ────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

ROOT = "/content/drive/MyDrive"
SOURCE = f"{ROOT}/mitbih/mamba_data.npz"          # 동결 atlas cohort
SYMBOLS = f"{ROOT}/mitbih/ecg_multi.npz"          # sym 을 가진 파일
RUNS = f"{ROOT}/MedKOS/ecg-model/runs"
PKGS = f"{ROOT}/mitbih/baseline_pkgs"             # 재분석용 baseline

for p in (SOURCE, SYMBOLS):
    print(("OK   " if os.path.exists(p) else "MISS "), p,
          (f"{os.path.getsize(p)/1e6:.1f} MB" if os.path.exists(p) else ""))
os.makedirs(RUNS, exist_ok=True)


In [ ]:
# ── 셀 4: RECOVER — 조인 + 음성대조군 + gate (재분석 없음) ──────────────────
import time
# 이 셀만 다시 돌렸을 때 stale 모듈에 걸리지 않도록 다시 확인한다
assert QB.MODULE_VERSION >= NEED_Q5B0, (
    f"stale module: q5b0 v{QB.MODULE_VERSION} < v{NEED_Q5B0} "
    f"({QB.__file__}). 셀 2를 다시 실행하고, 그래도 낮으면 BRANCH가 이 수정을 "
    "담고 있는지 확인한 뒤 Runtime > Restart runtime")
assert MODE == "RECOVER", f"이 셀은 RECOVER 전용 (지금 {MODE})"

log = QA.RunLog()
cohort, audit = QA.load_atlas_source(SOURCE, log=log)
rr = QA.rr_from_samples(cohort)
log(f"RR: {rr}")

STAMP = time.strftime("%Y%m%dT%H%M")
assert TIE_MODE in QB.TIE_MODES, f"TIE_MODE must be one of {QB.TIE_MODES}"
SLUG = QB.RUN_SLUG_B if TIE_MODE == QB.TIE_AGREEMENT else QB.RUN_SLUG
OUT = os.path.join(RUNS, QB.run_dir_name(STAMP, SLUG))
res = QB.run_recovery(cohort, SYMBOLS, OUT, tie_mode=TIE_MODE,
                      provenance={"atlas_source": audit,
                                  "notebook": "quest51_q5b0_subtype_key_recovery"},
                      log=log)
RECOVERY = res
print()
print("arm:", res["arm_id"], "| tie_mode:", res["tie_mode"])
print("gate:", res["gate"], "| status:", res["status"],
      "| training_performed:", res["training_performed"])
print(f"S beats joined : {res['n_matched']}/{res['n_s']} "
      f"({res['match_fraction']:.1%})")
print(f"symbol in A/a/J/S: {res['symbol_in_s_set_fraction']:.1%}"
      "   <- matcher는 symbol을 보지 않는다 (독립 검증)")
print("subtype counts :", res["subtype_counts"])
d = res.get("drop_map") or {}
if d.get("available"):
    print()
    print("cohort 전처리가 버린 beat:", d["n_missing_from_cohort"],
          f"/ {d['n_source']} ({d['missing_fraction']:.2%})")
    print("  클래스별:", d["missing_by_class"],
          "  <- S가 0이 아니면 Q5-A는 걸러진 S 집단을 채점한 것이다")
    print("  RR 오염 생존 beat 상한:", f"{d['rr_corruption_upper_bound']:.2%}")
    print("  최다:", d["worst_records"])
n = res.get("nearest_cost_unmatched") or {}
if n.get("n"):
    print()
    print("못 붙인 beat의 최근접 후보 거리: n=%d · p10 %.4fs · p50 %.4fs · p90 %.4fs"
          % (n["n"], n["p10"], n["p50"], n["p90"]))
    print("  허용치의 2배 안:", f"{n['within_2x_tolerance']:.1%}")
    print("  -> 0.005~0.05s 대에 퍼져 있으면 '검출 지터', 0.5~1.5s 대면 '이웃 소실',"
          " RR 전 범위에 균일하면 '식별 불가'")
if res["tie_mode"] == QB.TIE_AGREEMENT:
    print()
    print("동점 집합이 한 symbol로 일치해 부여된 beat:", res["n_tie_agreed"],
          f"(일치율 {res['tie_set_symbol_agreement']})")
    cov = res["subtype_coverage"]
    print("subtype별 회수율 (참값 대비):")
    for t, v in cov["by_subtype"].items():
        print(f"   {t}: {v['n_recovered']}/{v['n_true']}"
              + (f" = {v['coverage']:.3f}" if v["coverage"] is not None else "")
              + ("   <- 20박 미만, 참고용" if not v["scored"] else ""))
    print("  전체 회수율:", round(cov["overall_coverage"], 4),
          "| 최저/전체 비:", cov["worst_ratio_to_overall"],
          f"(≥ {cov['min_ratio']} 이어야 한다 — A만 회수하면 블록이 편향된다)")
print("bundle         :", OUT)


In [ ]:
# ── 셀 5: gate 확인 — NO_GO면 여기서 끝이고, 그것이 결론이다 ────────────────
assert MODE == "RECOVER", f"이 셀은 RECOVER 전용 (지금 {MODE})"

g = RECOVERY["gate_detail"]
for c in g["checks"]:
    print(f"  {'PASS' if c['pass'] else 'FAIL'}  {c['check']:38s} "
          f"{str(c['value']):>18s}   (needs {c['threshold']})")
print()
print("gate:", g["gate"])
print("next:", g["next_step"])
if not g["pass"]:
    print()
    print("NO_GO — B_SUBTYPE는 저장 산출물로 측정할 수 없다. 추정으로 채우지 않고,")
    print("이것을 만들려고 재학습하지 않는다. Q5-A의 UNRESOLVED(D5)는 4개 블록")
    print("위에서 그대로 유지된다. 셀 6은 실행하지 않는다.")
    for r in g["fail_reasons"]:
        print("  -", r)


In [ ]:
# ── 셀 6: REANALYZE — GO일 때만. Q5-A의 run_atlas를 그대로 다시 실행 ────────
import glob, json, time
# 이 셀만 다시 돌렸을 때 stale 모듈에 걸리지 않도록 다시 확인한다
assert QB.MODULE_VERSION >= NEED_Q5B0, (
    f"stale module: q5b0 v{QB.MODULE_VERSION} < v{NEED_Q5B0} "
    f"({QB.__file__}). 셀 2를 다시 실행하고, 그래도 낮으면 BRANCH가 이 수정을 "
    "담고 있는지 확인한 뒤 Runtime > Restart runtime")
assert MODE == "REANALYZE", f"이 셀은 REANALYZE 전용 (지금 {MODE})"

log = QA.RunLog()
cohort, audit = QA.load_atlas_source(SOURCE, log=log)
QA.rr_from_samples(cohort)

# 저장된 RECOVER bundle을 다시 읽는다 — notebook 변수에 기대지 않는다.
rec_dirs = sorted(glob.glob(f"{RUNS}/*_EXP-2026-005_q5b0_subtype_key_recovery"))
assert rec_dirs, "RECOVER를 먼저 실행한다"
RECOVERY = QB.load_recovery(rec_dirs[-1], cohort)      # cohort와 키가 어긋나면 STOP
print("recovery:", rec_dirs[-1], "| gate:", RECOVERY["gate"])
assert RECOVERY["gate_pass"], (
    "gate가 NO_GO다 — B_SUBTYPE는 종결이고 재분석하지 않는다")

# baseline은 Q5-A가 동결한 그대로 쓴다 (같은 freeze를 재사용해야 비교가 성립)
inv_dirs = sorted(glob.glob(f"{RUNS}/*_EXP-2026-004_q5a_inventory"))
assert inv_dirs, "Q5-A INVENTORY를 먼저 실행한다"
inv = json.load(open(f"{inv_dirs[-1]}/source_inventory.json", encoding="utf-8"))
freeze = json.load(open(f"{inv_dirs[-1]}/baseline_freeze.json", encoding="utf-8"))
print("freeze:", freeze["status"], sorted(freeze.get("selected", {})))
assert freeze["status"].startswith("FROZEN"), freeze.get("reasons")

source_index = QA.load_frozen_source_index(SOURCE, log=log)
models = {}
for label, sel in freeze.get("selected", {}).items():
    if not sel.get("beat_level_ready"):
        print(f"{label}: aggregate-only artifact, 제외")
        continue
    models[label] = QA.load_model_predictions(
        sel["run_dir"], label, source_index=source_index,
        arm=sel.get("model_name"), log=log)
assert models, "beat-level 산출물을 하나도 읽지 못했다"

STAMP = time.strftime("%Y%m%dT%H%M")
OUT2 = os.path.join(RUNS, QB.run_dir_name(STAMP, QB.REANALYSIS_SLUG))
result = QB.run_reanalysis(cohort, models, inv, freeze, RECOVERY, OUT2, log=log)

be = result.get("block_evidence", {})
print()
print("status:", result["status"], "| training_performed:",
      result["training_performed"])
print("blocks scored:", sorted(be.get("blocks", {})))
print("branch:", result["decision"]["branch"], f"({result['decision']['rule']})")
print("reason:", result["decision"]["reason"])
sc = result.get("subtype_shuffle_control", {})
print("subtype shuffle control:", {k: sc.get(k) for k in
                                   ("applicable", "real_delta",
                                    "shuffled_mean", "pass")})
for row in result.get("baseline_claim_check", []):
    if row.get("recorded_claim") is not None:
        print(f"claim {row['model']} {row['recorded_claim']}: {row['verdict']}")
print("bundle:", OUT2)


In [ ]:
# ── 셀 7: REPORT — 저장 bundle만 다시 표시 (재계산 없음) ────────────────────
# 이 셀만 다시 돌렸을 때 stale 모듈에 걸리지 않도록 다시 확인한다
assert QB.MODULE_VERSION >= NEED_Q5B0, (
    f"stale module: q5b0 v{QB.MODULE_VERSION} < v{NEED_Q5B0} "
    f"({QB.__file__}). 셀 2를 다시 실행하고, 그래도 낮으면 BRANCH가 이 수정을 "
    "담고 있는지 확인한 뒤 Runtime > Restart runtime")
assert MODE == "REPORT", f"이 셀은 REPORT 전용 (지금 {MODE})"

run = REPORT_RUN or sorted(
    [os.path.join(RUNS, d) for d in os.listdir(RUNS) if "EXP-2026-005" in d])[-1]
rep = QB.report_recovery(run)
print("run:", run)
print("status:", rep.get("status"))
print()
print(rep.get("summary", "(no summary)"))
